# LmrR EVB Preparation — Single-Point QM, RESP Charges, Barriers, Normal Modes

**What this notebook does, end to end:**
1. Generates fast **single-point DFT** inputs (production energies) on your
   already-optimized xTB RS/TS/PS geometries — no re-optimization, no DFT Hessian.
2. Generates **HF/6-31G\* single-point ESP inputs** for RESP charge fitting.
3. Parses the resulting QM output for **electronic energies**.
4. Reuses your **existing xTB Hessians/normal modes** (Stage 1) for
   ZPE + thermal corrections, combined with the new DFT electronic energy —
   this is the standard fast "DFT-electronic // xTB-thermal" composite scheme.
5. Fits **two-stage RESP charges** from the HF/6-31G* ESP grid (no AmberTools).
6. Assembles everything (charges, barriers, QM energies, normal-mode/ZPE data)
   into one summary table.

**Honest note on EVB-readiness, read before using this:**
Once this notebook finishes, you will have a properly-constructed QM
reference: RESP charges (correct convention), DFT single-point energies
(correct level), and ZPE/thermal-corrected barriers. This is exactly the
QM reference data an EVB parameterization is calibrated against — but it
is not itself an EVB free energy. The remaining step (outside this notebook)
is: build the classical valence-bond force field (RS/PS diabatic states with
these RESP charges), fit the off-diagonal coupling H12 and energy shift
alpha so the classical gas-phase EVB surface reproduces this QM barrier and
reaction energy, then run FEP/umbrella-sampling MD in the full protein+solvent
environment (e.g. with Q, CHARMM, or an OpenMM-based EVB implementation) to
get the actual free-energy barrier in the enzyme. This notebook produces the
calibration target, not the final EVB DeltaG.


## 0. One-time environment fix (numpy/scipy native crash)

The kernel has been crashing with Windows exit code `0xC06D007F` -- a native
(non-Python) crash signature typically caused by numpy and scipy being linked
against **mismatched or duplicated math libraries** (e.g. two copies of MKL,
or MKL + OpenBLAS both loaded at once). This can crash the kernel at
essentially random points, since the trigger is background math-library
thread initialization, not a specific line of your code.

**Run cell 0b below ONCE** to reinstall numpy/scipy from a single consistent
source, then **restart the kernel**, then run the rest of the notebook
normally from the top. You do not need to run cell 0b again after that --
comment it back out or skip it on future runs.

Cell 0a (the `KMP_DUPLICATE_LIB_OK` env var) is safe and cheap to leave in
and run every time -- it must execute before numpy is imported anywhere in
the kernel, which is why it's the very first cell in the notebook.

In [112]:
# --- 0a. Run this every time, FIRST, before any other cell ---
# Must be set before numpy/scipy are imported anywhere in this kernel session,
# since it controls how the OpenMP/MKL runtime handles a duplicate library
# load. This does not fix a genuine install conflict (see 0b) but prevents
# that conflict from being treated as fatal.
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"


In [113]:
# --- 0b. Run ONCE to fix the underlying numpy/scipy install conflict ---
# This reinstalls numpy and scipy from a single consistent source (whichever
# pip this kernel is using), removing any mixed conda+pip / duplicated-MKL
# install that's causing the 0xC06D007F native crash.
#
# After running this cell: RESTART THE KERNEL, then re-run the notebook from
# the top (cell 0a onward). You do not need to run this cell again afterward.

import sys

get_ipython().system(f'{sys.executable} -m pip uninstall numpy scipy -y')
get_ipython().system(f'{sys.executable} -m pip cache purge')
get_ipython().system(f'{sys.executable} -m pip install numpy scipy')

print()
print("Reinstall complete. RESTART THE KERNEL now (Kernel -> Restart), then run the notebook from the top.")


Found existing installation: numpy 2.5.2
Uninstalling numpy-2.5.2:
  Successfully uninstalled numpy-2.5.2
Found existing installation: scipy 1.18.1
Uninstalling scipy-1.18.1:
  Successfully uninstalled scipy-1.18.1
Files removed: 12 (49.9 MB)
Directories removed: 0
   ---------------------------------------- 0.0/12.5 MB ? eta -:--:--
   -- ------------------------------------- 0.8/12.5 MB 5.2 MB/s eta 0:00:03
   ------ --------------------------------- 2.1/12.5 MB 5.6 MB/s eta 0:00:02
   ---------- ----------------------------- 3.4/12.5 MB 5.7 MB/s eta 0:00:02
   -------------- ------------------------- 4.5/12.5 MB 5.4 MB/s eta 0:00:02
   ---------------- ----------------------- 5.2/12.5 MB 5.3 MB/s eta 0:00:02
   -------------------- ------------------- 6.3/12.5 MB 5.2 MB/s eta 0:00:02
   ------------------------ --------------- 7.6/12.5 MB 5.3 MB/s eta 0:00:01
   --------------------------- ------------ 8.7/12.5 MB 5.4 MB/s eta 0:00:01
   -------------------------------- ------- 10.2

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


### Optional: verify which BLAS backend numpy/scipy are actually using

Run this after restarting the kernel (post-reinstall) as a sanity check.
If this still shows more than one BLAS/LAPACK backend involved (e.g. both
`mkl_rt` and `openblas` referenced), that confirms a mixed install is still
present and the reinstall above needs to happen in a clean virtual
environment rather than the current one.

In [114]:
import numpy as np
import scipy
print("numpy:", np.__version__, "->", np.__file__)
print("scipy:", scipy.__version__, "->", scipy.__file__)
print()
np.show_config()


numpy: 2.5.2 -> c:\Users\Nayanika\miniconda3\envs\quimica-echem\Lib\site-packages\numpy\__init__.py
scipy: 1.18.1 -> c:\Users\Nayanika\miniconda3\envs\quimica-echem\Lib\site-packages\scipy\__init__.py

Build Dependencies:
  blas:
    detection method: pkgconfig
    found: true
    include directory: C:/Users/runneradmin/AppData/Local/Temp/cibw-run-b2ht6fnl/cp313-win_amd64/build/venv/Lib/site-packages/scipy_openblas64/include
    lib directory: C:/Users/runneradmin/AppData/Local/Temp/cibw-run-b2ht6fnl/cp313-win_amd64/build/venv/Lib/site-packages/scipy_openblas64/lib
    name: scipy-openblas
    openblas configuration: OpenBLAS 0.3.34.0.0  USE64BITINT DYNAMIC_ARCH NO_AFFINITY
      SkylakeX MAX_THREADS=24
    pc file directory: D:/a/numpy-release/numpy-release/.openblas
    version: 0.3.34.0.0
  lapack:
    detection method: pkgconfig
    found: true
    include directory: C:/Users/runneradmin/AppData/Local/Temp/cibw-run-b2ht6fnl/cp313-win_amd64/build/venv/Lib/site-packages/scipy_openbla

## 1. Imports and constants

In [115]:

from __future__ import annotations

import re
import json
import csv
from pathlib import Path
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
from scipy.optimize import minimize

ANGSTROM_TO_BOHR = 1.8897259886
HARTREE_TO_KCAL = 627.509474
HARTREE_TO_KJMOL = 2625.499638

# Physical constants for RRHO thermal corrections (SI, then converted).
H_PLANCK = 6.62607015e-34       # J s
K_BOLTZ = 1.380649e-23          # J/K
C_LIGHT_CM = 2.99792458e10      # cm/s
N_AVOGADRO = 6.02214076e23

# Bondi (1964) van der Waals radii (Angstrom) -- standard for MK/RESP grids.
VDW_RADII_ANGSTROM = {
    "H": 1.20, "C": 1.70, "N": 1.55, "O": 1.52, "F": 1.47,
    "P": 1.80, "S": 1.80, "CL": 1.75, "BR": 1.85, "I": 1.98, "SE": 1.90,
}

RESP_A_STAGE1 = 0.0005   # hartree, weak restraint, stage 1 (all atoms)
RESP_A_STAGE2 = 0.001    # hartree, stronger restraint, stage 2 (CH only)
RESP_B = 0.1             # hartree/e, hyperbolic restraint tightness


## 2. Configuration

Paste in (or import) the same `STRUCTURE_SETS` dict and `XTB_DIR` you used
in your Stage-1 xTB notebook, so this notebook reads the SAME optimized
geometries. Easiest: re-run/import the cell that defines `STRUCTURE_SETS`
from your original notebook before running this one, or save it once as
JSON from that notebook (`json.dump({k: vars(v) for k, v in STRUCTURE_SETS.items()}, ...)`)
and load it here.

### 2a. Structure-set definitions (self-contained copy)

This reproduces the exact `STRUCTURE_SETS` dict from your original Stage-1 notebook so this notebook runs standalone. **If you've since edited the step definitions, bond changes, or added new steps in your original notebook, update this cell to match** -- this is a static copy, not a live import.

In [116]:
# ============================================================
# Structure-set definitions (copied from your Stage-1 notebook)
# ============================================================
from dataclasses import dataclass

@dataclass(frozen=True)
class BondChange:
    kind: str
    serial_i: int
    serial_j: int
    note: str

@dataclass(frozen=True)
class StructureSet:
    label: str
    rs_pdb: str
    ts_pdb: str
    ps_pdb: str
    charge: int
    multiplicity: int
    bond_changes: tuple
    comment: str

STRUCTURE_SETS = {
    "RS1_1_to_TS1_2": StructureSet(
        label="RS1_1_to_TS1_2",
        rs_pdb="step_1_1_RS.pdb",
        ts_pdb="step_1_1_TS.pdb",
        ps_pdb="step_1_1_PS.pdb",
        charge=0,
        multiplicity=1,
        bond_changes=(
            BondChange("formed", 1, 10, "PAF.N2 attacks ENL.C10; N-C bond formation"),
            BondChange("weakened", 10, 16, "ENL.C10=O1 carbonyl weakens"),
        ),
        comment="Step 1.1 addition. Use TS as a starting TS guess; PS is the tetrahedral state.",
    ),
    "TS1_2_to_PS1_2b": StructureSet(
        label="TS1_2_to_PS1_2b",
        rs_pdb="step_1_2_RS.pdb",
        ts_pdb="step_1_2_TS.pdb",
        ps_pdb="step_1_2_PS.pdb",
        charge=0,
        multiplicity=1,
        bond_changes=(
            BondChange("broken", 1, 29, "PAF N-H breaks"),
            BondChange("formed", 27, 29, "W1 accepts H29"),
            BondChange("broken", 60, 61, "W2 O-H breaks"),
            BondChange("formed", 16, 61, "ENL.O1 accepts H61"),
        ),
        comment="Two-water proton redistribution.",
    ),
    "RS1_2b_to_PS1_3": StructureSet(
        label="RS1_2b_to_PS1_3",
        rs_pdb="step_1_3_RS.pdb",
        ts_pdb="step_1_3_TS.pdb",
        ps_pdb="step_1_3_PS.pdb",
        charge=0,
        multiplicity=1,
        bond_changes=(
            BondChange("broken", 27, 29, "W1 hydronium O-H breaks"),
            BondChange("formed", 16, 29, "carbinolamine O accepts H29"),
            BondChange("broken", 10, 16, "C-O leaving-water bond breaks"),
            BondChange("double-bond formed", 1, 10, "iminium N=C forms"),
        ),
        comment="Dehydration to iminium.",
    ),
    "RS2_1_to_TS2_1a": StructureSet(
        label="RS2_1_to_TS2_1a",
        rs_pdb="step_2_1_RS.pdb",
        ts_pdb="step_2_1_TS.pdb",
        ps_pdb="step_2_1_PS.pdb",
        charge=0,
        multiplicity=1,
        bond_changes=(
            BondChange("formed", 12, 20, "new C-C sigma bond ENL.C12--IND.C3"),
            BondChange("weakened", 1, 10, "iminium N=C weakens"),
            BondChange("strengthened", 10, 11, "C10-C11 bond strengthens"),
        ),
        comment="Friedel-Crafts C-C bond formation.",
    ),
    "TS2_1a_to_PS2_2": StructureSet(
        label="TS2_1a_to_PS2_2",
        rs_pdb="step_2_2a_RS.pdb",
        ts_pdb="step_2_2b_TS.pdb",
        ps_pdb="step_2_2b_PS.pdb",
        charge=0,
        multiplicity=1,
        bond_changes=(
            BondChange("broken", 20, 50, "IND.C3-H50 breaks"),
            BondChange("formed", 60, 50, "W2 accepts H50"),
            BondChange("broken", 27, 58, "W1 O-H58 breaks"),
            BondChange("formed", 11, 58, "ENL.C11 accepts H58"),
            BondChange("double-bond formed", 1, 10, "N=C/enamine bond order changes"),
        ),
        comment="Combined 2.2 tautomerization. If you want separate 2.2a/2.2b, split this entry.",
    ),
}

print(f"Loaded {len(STRUCTURE_SETS)} structure sets:", list(STRUCTURE_SETS))


Loaded 5 structure sets: ['RS1_1_to_TS1_2', 'TS1_2_to_PS1_2b', 'RS1_2b_to_PS1_3', 'RS2_1_to_TS2_1a', 'TS2_1a_to_PS2_2']


In [117]:

# --- EDIT THESE FOUR PATHS/VALUES FOR YOUR MACHINE ---

XTB_DIR = Path(r"D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_5QM\xtb_preparation")
STAGE2_XTB_DIR = XTB_DIR / "stage2_five_xtb_protocols"   # your existing xTB Hessian/freq outputs
SP_OUT_DIR = XTB_DIR.parent / "single_point_qm"           # new: SP + RESP inputs go here
SUMMARY_OUT = XTB_DIR.parent / "evb_reference_summary"
SUMMARY_OUT.mkdir(parents=True, exist_ok=True)

# If STRUCTURE_SETS is already defined earlier in your kernel session
# (e.g. you ran the original Stage-1 notebook cells first), this notebook
# will just reuse it. Otherwise, load it from a JSON export:
#
# with open("structure_sets.json") as f:
#     raw = json.load(f)
# from dataclasses import make_dataclass
# ... reconstruct STRUCTURE_SETS here ...

assert "STRUCTURE_SETS" in globals(), (
    "STRUCTURE_SETS is not defined. Run your Stage-1 notebook's definition "
    "cell first, or load it from a saved JSON export."
)

STEPS = list(STRUCTURE_SETS)
ROLES = ("RS", "TS", "PS")
print(f"{len(STEPS)} reaction steps x {len(ROLES)} roles = {len(STEPS)*len(ROLES)} structures")


5 reaction steps x 3 roles = 15 structures


## 3. Geometry I/O (matches your existing pipeline's XYZ convention)

In [118]:

def read_xyz(path: Path) -> tuple[list[str], list[tuple[float, float, float]], str]:
    lines = Path(path).read_text(encoding="utf-8").splitlines()
    n = int(lines[0].strip())
    comment = lines[1] if len(lines) > 1 else ""
    labels: list[str] = []
    coords: list[tuple[float, float, float]] = []
    for line in lines[2 : 2 + n]:
        parts = line.split()
        labels.append(parts[0])
        coords.append((float(parts[1]), float(parts[2]), float(parts[3])))
    return labels, coords, comment


## 4. Single-point DFT (production) + RESP (HF/6-31G*) input writers

In [119]:
import os

def write_orca_sp_production(
    inp_path: Path, xyz_path: Path, charge: int, multiplicity: int,
    functional: str = "PBE0", basis: str = "def2-TZVP",
    dispersion: str = "D3BJ", use_rijcosx: bool = True,
    scf_convergence: str = "TightSCF",   # use "SCF" (default/looser) for a faster, less precise pass
    nprocs: int = 8, memory_mb: int = 4000,
) -> None:
    labels, coords, _ = read_xyz(xyz_path)
    route_parts = ["!", functional, basis, dispersion, scf_convergence, "SP"]
    if use_rijcosx:
        route_parts += ["RIJCOSX", "def2/J"]
    lines = [
        " ".join(route_parts),
        f"%pal nprocs {nprocs} end",
        f"%maxcore {memory_mb}",
        "",
        f"* xyz {charge} {multiplicity}",
    ]
    for label, (x, y, z) in zip(labels, coords):
        lines.append(f"  {label:<2s} {x:14.8f} {y:14.8f} {z:14.8f}")
    lines += ["*", ""]
    inp_path.parent.mkdir(parents=True, exist_ok=True)
    inp_path.write_text("\n".join(lines), encoding="utf-8")


def write_orca_sp_resp(
    inp_path: Path, xyz_path: Path, charge: int, multiplicity: int,
    nprocs: int = 8, memory_mb: int = 4000,
) -> None:
    labels, coords, _ = read_xyz(xyz_path)
    lines = [
        "! HF 6-31G* TightSCF SP",
        f"%pal nprocs {nprocs} end",
        f"%maxcore {memory_mb}",
        "%elprop",
        "  Dipole true",
        "end",
        "",
        f"* xyz {charge} {multiplicity}",
    ]
    for label, (x, y, z) in zip(labels, coords):
        lines.append(f"  {label:<2s} {x:14.8f} {y:14.8f} {z:14.8f}")
    lines += ["*", ""]
    inp_path.parent.mkdir(parents=True, exist_ok=True)
    inp_path.write_text("\n".join(lines), encoding="utf-8")


# Two presets for the production single point:
#   "accurate" -- def2-TZVP, TightSCF: what you'd report in the paper.
#   "fast"     -- def2-SVP, default SCF convergence: for a quick approximate
#                 barrier now. Basis-set error partly cancels between RS/TS/PS
#                 of the SAME step (same functional, same molecule), so
#                 relative barriers from "fast" are usually a reasonable
#                 first look -- but re-run "accurate" before trusting numbers
#                 for publication. Typically several times faster than TZVP,
#                 mainly because it's a much smaller basis (fewer functions
#                 for RIJCOSX/SCF to work with), not because RIJCOSX itself
#                 changes -- RIJCOSX is already on in both presets.
SP_PRESETS = {
    "accurate": dict(basis="def2-TZVP", scf_convergence="TightSCF"),
    "fast":     dict(basis="def2-SVP",  scf_convergence="SCF"),
}


def generate_all_sp_inputs(steps, structure_sets, xtb_dir, out_dir,
                            roles=("RS", "TS", "PS"), functional="PBE0",
                            preset: str = "accurate", nprocs: int = None):
    if nprocs is None:
        nprocs = max(1, os.cpu_count() or 1)  # auto-detect actual cores instead of hardcoding 8
    preset_kwargs = SP_PRESETS[preset]
    manifest = []
    for step in steps:
        cfg = structure_sets[step]
        for role in roles:
            xyz_path = xtb_dir / step / f"{step}_{role}.xyz"
            if not xyz_path.exists():
                raise FileNotFoundError(f"Missing xTB-optimized geometry: {xyz_path}")
            step_out = out_dir / step
            basis_tag = preset_kwargs["basis"].replace("-", "")
            prod_inp = step_out / f"{step}_{role}_sp_{functional}_{basis_tag}.inp"
            resp_inp = step_out / f"{step}_{role}_resp_hf631gs.inp"
            write_orca_sp_production(prod_inp, xyz_path, cfg.charge, cfg.multiplicity,
                                      functional=functional, nprocs=nprocs, **preset_kwargs)
            write_orca_sp_resp(resp_inp, xyz_path, cfg.charge, cfg.multiplicity, nprocs=nprocs)
            manifest.append({
                "step": step, "role": role, "xyz_source": str(xyz_path),
                "charge": cfg.charge, "multiplicity": cfg.multiplicity,
                "orca_production": str(prod_inp), "orca_resp": str(resp_inp),
                "preset": preset, "nprocs": nprocs,
            })
    return pd.DataFrame(manifest)


In [120]:
# Set preset="fast" to get an approximate barrier quickly (def2-SVP, looser
# SCF convergence). Switch to preset="accurate" (def2-TZVP, TightSCF) once
# you have time, before trusting numbers for publication. nprocs is
# auto-detected from this machine's actual core count unless you pass one
# explicitly.
SP_PRESET = "fast"   # <-- "fast" or "accurate"

print(f"Detected {os.cpu_count()} CPU cores -- using that many for %pal nprocs unless overridden.")

sp_manifest = generate_all_sp_inputs(
    steps=STEPS, structure_sets=STRUCTURE_SETS, xtb_dir=XTB_DIR,
    out_dir=SP_OUT_DIR, functional="PBE0", preset=SP_PRESET,
)
sp_manifest.to_csv(SUMMARY_OUT / "sp_and_resp_input_manifest.csv", index=False)
sp_manifest


Detected 8 CPU cores -- using that many for %pal nprocs unless overridden.


,step,role,xyz_source,charge,multiplicity,orca_production,orca_resp,preset,nprocs
0,RS1_1_to_TS1_2,RS,D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_5QM\x...,0,1,D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_5QM\s...,D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_5QM\s...,fast,8
1,RS1_1_to_TS1_2,TS,D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_5QM\x...,0,1,D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_5QM\s...,D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_5QM\s...,fast,8
2,RS1_1_to_TS1_2,PS,D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_5QM\x...,0,1,D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_5QM\s...,D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_5QM\s...,fast,8
3,TS1_2_to_PS1_2b,RS,D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_5QM\x...,0,1,D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_5QM\s...,D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_5QM\s...,fast,8
4,TS1_2_to_PS1_2b,TS,D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_5QM\x...,0,1,D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_5QM\s...,D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_5QM\s...,fast,8
5,TS1_2_to_PS1_2b,PS,D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_5QM\x...,0,1,D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_5QM\s...,D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_5QM\s...,fast,8
6,RS1_2b_to_PS1_3,RS,D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_5QM\x...,0,1,D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_5QM\s...,D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_5QM\s...,fast,8
7,RS1_2b_to_PS1_3,TS,D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_5QM\x...,0,1,D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_5QM\s...,D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_5QM\s...,fast,8
8,RS1_2b_to_PS1_3,PS,D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_5QM\x...,0,1,D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_5QM\s...,D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_5QM\s...,fast,8
9,RS2_1_to_TS2_1a,RS,D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_5QM\x...,0,1,D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_5QM\s...,D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_5QM\s...,fast,8


## 5. Run the QM jobs externally

Submit the `_sp_PBE0.inp` and `_resp_hf631gs.inp` files listed above to ORCA.
These are single-point-only jobs (no Opt, no Freq), so each should take
minutes, not days. Come back to this notebook once they've finished and
point `SP_OUT_DIR` at the completed `.out` files (same directory, ORCA
writes `<inputname>.out` next to the `.inp` by default).

### 5a. Check: which of the 30 ORCA jobs still need to run

Fast, read-only -- just checks file existence and looks for ORCA's own
"TERMINATED NORMALLY" marker in each `.out` file. Run this any time to see
where you stand without touching anything.

In [121]:
def _orca_job_status(inp_path: Path) -> dict:
    inp_path = Path(inp_path)
    out_path = inp_path.with_suffix(".out")
    status = {"inp_exists": inp_path.exists(), "out_exists": out_path.exists(), "done": False}
    if out_path.exists():
        text = out_path.read_text(encoding="utf-8", errors="replace")
        status["done"] = "****ORCA TERMINATED NORMALLY****" in text
        status["out_has_error"] = ("ORCA finished by error termination" in text) or (not status["done"] and "ORCA TERMINATED" in text)
    return status

job_rows = []
for _, r in sp_manifest.iterrows():
    prod = _orca_job_status(r["orca_production"])
    resp = _orca_job_status(r["orca_resp"])
    job_rows.append({
        "step": r["step"], "role": r["role"],
        "production_inp": prod["inp_exists"], "production_out": prod["out_exists"], "production_done": prod["done"],
        "resp_inp": resp["inp_exists"], "resp_out": resp["out_exists"], "resp_done": resp["done"],
    })

job_status_df = pd.DataFrame(job_rows)
n_prod_pending = (~job_status_df["production_done"]).sum()
n_resp_pending = (~job_status_df["resp_done"]).sum()
print(f"Production SP jobs: {len(job_status_df) - n_prod_pending}/{len(job_status_df)} done, {n_prod_pending} pending")
print(f"RESP SP jobs:       {len(job_status_df) - n_resp_pending}/{len(job_status_df)} done, {n_resp_pending} pending")
job_status_df


Production SP jobs: 0/15 done, 15 pending
RESP SP jobs:       0/15 done, 15 pending


,step,role,production_inp,production_out,production_done,resp_inp,resp_out,resp_done
0,RS1_1_to_TS1_2,RS,True,False,False,True,False,False
1,RS1_1_to_TS1_2,TS,True,False,False,True,False,False
2,RS1_1_to_TS1_2,PS,True,False,False,True,False,False
3,TS1_2_to_PS1_2b,RS,True,False,False,True,False,False
4,TS1_2_to_PS1_2b,TS,True,False,False,True,False,False
5,TS1_2_to_PS1_2b,PS,True,False,False,True,False,False
6,RS1_2b_to_PS1_3,RS,True,False,False,True,False,False
7,RS1_2b_to_PS1_3,TS,True,False,False,True,False,False
8,RS1_2b_to_PS1_3,PS,True,False,False,True,False,False
9,RS2_1_to_TS2_1a,RS,True,False,False,True,False,False


### 5b. Run: submit the pending jobs to ORCA locally

**Off by default.** Set `RUN_ORCA_JOBS = True` below and set `ORCA_EXE` to
the full path of your `orca.exe` before running this cell -- ORCA's own docs
recommend always calling it by full path rather than relying on PATH,
especially for parallel (`%pal`) jobs.

Runs sequentially, one job at a time, skipping anything `5a` already found
done. Each `.inp` already has its own `%pal nprocs ...` block, so this does
not parallelize *across* jobs -- if you want several jobs running at once
(one per structure), you'd need to launch this in multiple notebook/terminal
sessions with disjoint subsets, which risks oversubscribing CPUs if not
sized carefully; sequential is the safe default here.

These are single-point-only jobs (no Opt, no Freq) so each should take
minutes -- if one runs far longer than that, something is likely wrong with
that input (check its `.out` as it grows).

In [122]:
import subprocess

RUN_ORCA_JOBS = False   # <-- flip to True once ORCA_EXE below is correct
ORCA_EXE = r"C:\Path\To\orca.exe"   # <-- EDIT: full path to your orca executable

def _run_orca(inp_path: Path, orca_exe: str) -> bool:
    inp_path = Path(inp_path)
    out_path = inp_path.with_suffix(".out")
    print(f"  running: {inp_path.name} ...", flush=True)
    with open(out_path, "w", encoding="utf-8") as out_f:
        result = subprocess.run([orca_exe, str(inp_path)], stdout=out_f, stderr=subprocess.STDOUT)
    ok = result.returncode == 0
    print(f"    {'OK' if ok else 'FAILED (returncode ' + str(result.returncode) + ')'} -> {out_path.name}")
    return ok

if not RUN_ORCA_JOBS:
    print("RUN_ORCA_JOBS is False -- not submitting anything. "
          "Set RUN_ORCA_JOBS = True and check ORCA_EXE to actually run jobs.")
else:
    if not Path(ORCA_EXE).exists():
        raise FileNotFoundError(f"ORCA_EXE not found at {ORCA_EXE} -- set the correct full path first.")

    pending = []
    for _, r in sp_manifest.iterrows():
        if not _orca_job_status(r["orca_production"])["done"]:
            pending.append(("production", r["step"], r["role"], r["orca_production"]))
        if not _orca_job_status(r["orca_resp"])["done"]:
            pending.append(("resp", r["step"], r["role"], r["orca_resp"]))

    print(f"{len(pending)} job(s) pending -- running sequentially...\n")
    run_log = []
    for kind, step, role, inp_path in pending:
        ok = _run_orca(inp_path, ORCA_EXE)
        run_log.append({"kind": kind, "step": step, "role": role, "inp": str(inp_path), "ok": ok})

    run_log_df = pd.DataFrame(run_log)
    run_log_df.to_csv(SUMMARY_OUT / "orca_run_log.csv", index=False)
    print()
    print(f"Done. {run_log_df['ok'].sum()}/{len(run_log_df)} succeeded. "
          f"Log written to {SUMMARY_OUT / 'orca_run_log.csv'}. Re-run cell 5a to confirm.")
    run_log_df


RUN_ORCA_JOBS is False -- not submitting anything. Set RUN_ORCA_JOBS = True and check ORCA_EXE to actually run jobs.


### 5c. Run: `orca_vpot` on finished RESP jobs (needed before section 10 can fit charges)

Also off by default. This is the step that actually produces the
`_vpot_out.txt` files that `resp_charges.csv` is currently reporting as
missing. Requires:
1. The RESP `.out` job to have finished (`resp_done = True` in `5a`).
2. A grid file (`..._grid.txt`) -- if it doesn't exist yet, this cell
   generates it from the structure's geometry the same way section 10 does.

**Verify the `orca_vpot` argument order below against `orca_vpot` (no args)
for your installed ORCA version before trusting this** -- it varies between
versions, and this is the one piece of the pipeline that was written from
the ORCA manual rather than confirmed against your actual install.

In [123]:
RUN_ORCA_VPOT = False   # <-- flip to True once you've verified the orca_vpot syntax below
ORCA_VPOT_EXE = str(Path(ORCA_EXE).with_name("orca_vpot"))  # usually sits next to orca.exe

vpot_rows = []
for _, r in sp_manifest.iterrows():
    step, role = r["step"], r["role"]
    resp_inp = Path(r["orca_resp"])
    resp_out = resp_inp.with_suffix(".out")
    gbw_path = resp_inp.with_suffix(".gbw")
    scfp_path = resp_inp.with_suffix(".scfp")
    grid_file = resp_inp.with_name(f"{step}_{role}_grid.txt")
    vpot_file = resp_inp.with_name(f"{step}_{role}_vpot_out.txt")

    row = {"step": step, "role": role, "attempted": False, "ok": False, "note": ""}

    if not (resp_out.exists() and _orca_job_status(resp_inp)["done"]):
        row["note"] = "RESP job not finished yet -- run 5b first"
        vpot_rows.append(row)
        continue
    if vpot_file.exists():
        row["note"] = "vpot output already exists"
        vpot_rows.append(row)
        continue
    if not grid_file.exists():
        labels, coords, _ = read_xyz(Path(r["xyz_source"]))
        grid = generate_mk_esp_grid(labels, coords)
        write_grid_for_orca_vpot(grid, grid_file)

    if not RUN_ORCA_VPOT:
        row["note"] = "RUN_ORCA_VPOT is False -- not running"
        vpot_rows.append(row)
        continue
    if not gbw_path.exists():
        row["note"] = f"no .gbw found next to {resp_inp.name} -- check RESP job actually wrote one"
        vpot_rows.append(row)
        continue

    row["attempted"] = True
    cmd = [ORCA_VPOT_EXE, str(gbw_path), str(scfp_path), str(grid_file), str(vpot_file)]
    print(f"  orca_vpot: {step}/{role} ...", flush=True)
    result = subprocess.run(cmd, capture_output=True, text=True)
    row["ok"] = result.returncode == 0 and vpot_file.exists()
    if not row["ok"]:
        row["note"] = f"returncode={result.returncode}; stderr: {result.stderr[:300]}"
        print(f"    FAILED: {row['note']}")
    else:
        print(f"    OK -> {vpot_file.name}")
    vpot_rows.append(row)

vpot_log_df = pd.DataFrame(vpot_rows)
vpot_log_df


,step,role,attempted,ok,note
0,RS1_1_to_TS1_2,RS,False,False,RESP job not finished yet -- run 5b first
1,RS1_1_to_TS1_2,TS,False,False,RESP job not finished yet -- run 5b first
2,RS1_1_to_TS1_2,PS,False,False,RESP job not finished yet -- run 5b first
3,TS1_2_to_PS1_2b,RS,False,False,RESP job not finished yet -- run 5b first
4,TS1_2_to_PS1_2b,TS,False,False,RESP job not finished yet -- run 5b first
5,TS1_2_to_PS1_2b,PS,False,False,RESP job not finished yet -- run 5b first
6,RS1_2b_to_PS1_3,RS,False,False,RESP job not finished yet -- run 5b first
7,RS1_2b_to_PS1_3,TS,False,False,RESP job not finished yet -- run 5b first
8,RS1_2b_to_PS1_3,PS,False,False,RESP job not finished yet -- run 5b first
9,RS2_1_to_TS2_1a,RS,False,False,RESP job not finished yet -- run 5b first


## 6. Parse ORCA single-point output for electronic energy

In [124]:

def parse_orca_final_energy(out_path: Path) -> float | None:
    '''Returns the FINAL SINGLE POINT ENERGY in Hartree, or None if not found
    (e.g. job still running or crashed -- check the .out file manually).'''
    text = Path(out_path).read_text(encoding="utf-8", errors="replace")
    matches = re.findall(r"FINAL SINGLE POINT ENERGY\s+(-?\d+\.\d+)", text)
    if not matches:
        return None
    return float(matches[-1])


def collect_production_energies(manifest: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, r in manifest.iterrows():
        out_path = Path(r["orca_production"]).with_suffix(".out")
        energy = parse_orca_final_energy(out_path) if out_path.exists() else None
        rows.append({
            "step": r["step"], "role": r["role"],
            "orca_out": str(out_path), "found": out_path.exists(),
            "E_DFT_hartree": energy,
        })
    return pd.DataFrame(rows)


In [125]:

dft_energies = collect_production_energies(sp_manifest)
missing = dft_energies[dft_energies["E_DFT_hartree"].isna()]
if len(missing):
    print(f"{len(missing)} jobs not yet finished / not found:")
    display(missing[["step", "role", "orca_out"]])
dft_energies


15 jobs not yet finished / not found:


,step,role,orca_out
0,RS1_1_to_TS1_2,RS,D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_5QM\s...
1,RS1_1_to_TS1_2,TS,D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_5QM\s...
2,RS1_1_to_TS1_2,PS,D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_5QM\s...
3,TS1_2_to_PS1_2b,RS,D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_5QM\s...
4,TS1_2_to_PS1_2b,TS,D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_5QM\s...
5,TS1_2_to_PS1_2b,PS,D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_5QM\s...
6,RS1_2b_to_PS1_3,RS,D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_5QM\s...
7,RS1_2b_to_PS1_3,TS,D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_5QM\s...
8,RS1_2b_to_PS1_3,PS,D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_5QM\s...
9,RS2_1_to_TS2_1a,RS,D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_5QM\s...


,step,role,orca_out,found,E_DFT_hartree
0,RS1_1_to_TS1_2,RS,D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_5QM\s...,False,None
1,RS1_1_to_TS1_2,TS,D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_5QM\s...,False,None
2,RS1_1_to_TS1_2,PS,D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_5QM\s...,False,None
3,TS1_2_to_PS1_2b,RS,D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_5QM\s...,False,None
4,TS1_2_to_PS1_2b,TS,D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_5QM\s...,False,None
5,TS1_2_to_PS1_2b,PS,D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_5QM\s...,False,None
6,RS1_2b_to_PS1_3,RS,D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_5QM\s...,False,None
7,RS1_2b_to_PS1_3,TS,D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_5QM\s...,False,None
8,RS1_2b_to_PS1_3,PS,D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_5QM\s...,False,None
9,RS2_1_to_TS2_1a,RS,D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_5QM\s...,False,None


## 7. Reuse existing xTB normal modes for ZPE + thermal correction

Re-running a DFT Hessian is the expensive part you're trying to avoid.
Standard practice: keep the DFT *electronic* energy from the fast
single-point above, but take the zero-point energy and thermal (RRHO)
corrections from your already-computed xTB Hessian (Stage 1/2). This
"DFT-electronic // xTB-thermal" composite is a common, defensible speed
trade-off — just say so explicitly in your methods section.

This cell expects your existing Stage-2 frequency CSV (the one your
original notebook already writes: vibrational frequencies in cm^-1 per
step/role). Point `FREQ_CSV` at that file.

In [126]:
# ------------------------------------------------------------
# Your actual Stage-2 frequency files live one per protocol/step/role:
#   STAGE2_DIR / <protocol> / <step> / "<step>_<role>_frequencies.csv"
# with columns: mode, frequency_cm-1, imaginary
# (confirmed from your original Stage-2 xTB run cell -- not a guess).
#
# Since you have 5 protocols (GFN1_tight, GFN2_tight, GFN2_etemp300,
# GFN2_etemp500, GFN2_normal), pick ONE as your ZPE/thermal-correction
# source. GFN2_tight is the most defensible choice (tight convergence,
# GFN2 is more accurate than GFN1, no electronic-temperature smearing
# unless your system needs it for near-degenerate states).
# ------------------------------------------------------------

FREQ_PROTOCOL = "GFN2_tight"   # <-- change if you prefer a different one

def load_xtb_frequencies_for_step_role(step: str, role: str) -> pd.DataFrame:
    freq_file = STAGE2_XTB_DIR / FREQ_PROTOCOL / step / f"{step}_{role}_frequencies.csv"
    if not freq_file.exists():
        raise FileNotFoundError(f"Missing frequency file: {freq_file}")
    df = pd.read_csv(freq_file)
    df = df.rename(columns={"frequency_cm-1": "frequency_cm1"})
    df["step"] = step
    df["role"] = role
    return df

def load_all_xtb_frequencies(steps, roles=("RS", "TS", "PS")) -> pd.DataFrame:
    frames = []
    for step in steps:
        for role in roles:
            frames.append(load_xtb_frequencies_for_step_role(step, role))
    return pd.concat(frames, ignore_index=True)

In [127]:
# Drop-in replacement: scan instead of crash-on-first-miss
def audit_xtb_frequency_files(steps, roles=("RS", "TS", "PS")):
    rows = []
    for step in steps:
        step_dir = STAGE2_XTB_DIR / FREQ_PROTOCOL / step
        for role in roles:
            expected = step_dir / f"{step}_{role}_frequencies.csv"
            rows.append({
                "step": step, "role": role,
                "expected_file": str(expected),
                "exists": expected.exists(),
            })
    df = pd.DataFrame(rows)
    print(df[~df["exists"]].to_string(index=False) if (~df["exists"]).any() else "All files present.")
    # Also show what's ACTUALLY in each step's folder, to catch naming mismatches
    for step in steps:
        step_dir = STAGE2_XTB_DIR / FREQ_PROTOCOL / step
        if step_dir.exists():
            actual = sorted(p.name for p in step_dir.glob("*frequencies*"))
            missing_here = df[(df["step"] == step) & (~df["exists"])]
            if len(missing_here):
                print(f"\n{step} — files actually present in {step_dir}:")
                print(actual or "  (none matching *frequencies*)")
        else:
            print(f"\n{step} — directory does not exist: {step_dir}")
    return df

audit_df = audit_xtb_frequency_files(STEPS, ROLES)

           step role                                                                                                                                        expected_file  exists
TS1_2_to_PS1_2b   PS D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_5QM\xtb_preparation\stage2_five_xtb_protocols\GFN2_tight\TS1_2_to_PS1_2b\TS1_2_to_PS1_2b_PS_frequencies.csv   False
RS1_2b_to_PS1_3   RS D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_5QM\xtb_preparation\stage2_five_xtb_protocols\GFN2_tight\RS1_2b_to_PS1_3\RS1_2b_to_PS1_3_RS_frequencies.csv   False
RS1_2b_to_PS1_3   TS D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_5QM\xtb_preparation\stage2_five_xtb_protocols\GFN2_tight\RS1_2b_to_PS1_3\RS1_2b_to_PS1_3_TS_frequencies.csv   False
RS1_2b_to_PS1_3   PS D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_5QM\xtb_preparation\stage2_five_xtb_protocols\GFN2_tight\RS1_2b_to_PS1_3\RS1_2b_to_PS1_3_PS_frequencies.csv   False
RS2_1_to_TS2_1a   RS D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_5QM\xtb_preparation\stage2_five_xtb_protocols\GF

In [128]:
def check_across_protocols(steps_roles, stage2_dir):
    protocols = ["GFN1_tight", "GFN2_tight", "GFN2_etemp300", "GFN2_etemp500", "GFN2_normal"]
    rows = []
    for step, role in steps_roles:
        for proto in protocols:
            f = stage2_dir / proto / step / f"{step}_{role}_frequencies.csv"
            rows.append({"step": step, "role": role, "protocol": proto, "exists": f.exists()})
    return pd.DataFrame(rows)

gaps = [
    ("TS1_2_to_PS1_2b", "PS"),
    ("RS1_2b_to_PS1_3", "RS"), ("RS1_2b_to_PS1_3", "TS"), ("RS1_2b_to_PS1_3", "PS"),
    ("RS2_1_to_TS2_1a", "RS"), ("RS2_1_to_TS2_1a", "TS"), ("RS2_1_to_TS2_1a", "PS"),
]
protocol_check = check_across_protocols(gaps, STAGE2_XTB_DIR)
print(protocol_check.pivot(index=["step","role"], columns="protocol", values="exists"))

protocol              GFN1_tight  GFN2_etemp300  GFN2_etemp500  GFN2_normal  \
step            role                                                          
RS1_2b_to_PS1_3 PS         False          False           True        False   
                RS          True          False           True        False   
                TS         False           True           True        False   
RS2_1_to_TS2_1a PS         False          False           True        False   
                RS         False          False           True        False   
                TS         False          False           True        False   
TS1_2_to_PS1_2b PS         False          False           True        False   

protocol              GFN2_tight  
step            role              
RS1_2b_to_PS1_3 PS         False  
                RS         False  
                TS         False  
RS2_1_to_TS2_1a PS         False  
                RS         False  
                TS         False  
TS1_2_to

In [129]:
for step, role in gaps:
    xyz = XTB_DIR / step / f"{step}_{role}.xyz"
    print(step, role, "geometry exists:", xyz.exists())

TS1_2_to_PS1_2b PS geometry exists: True
RS1_2b_to_PS1_3 RS geometry exists: True
RS1_2b_to_PS1_3 TS geometry exists: True
RS1_2b_to_PS1_3 PS geometry exists: True
RS2_1_to_TS2_1a RS geometry exists: True
RS2_1_to_TS2_1a TS geometry exists: True
RS2_1_to_TS2_1a PS geometry exists: True


In [130]:
FREQ_PROTOCOL = "GFN2_tight"   # default for everything that converges here

# Steps/roles where GFN2_tight's Hessian SCF failed to converge (near-degenerate
# frontier orbitals from proton-transfer/charge-delocalized TS-like structures);
# GFN2_etemp500 was the only protocol that converged cleanly for these.
FREQ_PROTOCOL_OVERRIDES = {
    ("TS1_2_to_PS1_2b", "PS"): "GFN2_etemp500",
    ("RS1_2b_to_PS1_3", "RS"): "GFN2_etemp500",
    ("RS1_2b_to_PS1_3", "TS"): "GFN2_etemp500",
    ("RS1_2b_to_PS1_3", "PS"): "GFN2_etemp500",
    ("RS2_1_to_TS2_1a", "RS"): "GFN2_etemp500",
    ("RS2_1_to_TS2_1a", "TS"): "GFN2_etemp500",
    ("RS2_1_to_TS2_1a", "PS"): "GFN2_etemp500",
}

def load_xtb_frequencies_for_step_role(step: str, role: str) -> pd.DataFrame:
    protocol = FREQ_PROTOCOL_OVERRIDES.get((step, role), FREQ_PROTOCOL)
    freq_file = STAGE2_XTB_DIR / protocol / step / f"{step}_{role}_frequencies.csv"
    if not freq_file.exists():
        raise FileNotFoundError(f"Missing frequency file: {freq_file}")
    df = pd.read_csv(freq_file)
    df = df.rename(columns={"frequency_cm-1": "frequency_cm1"})
    df["step"] = step
    df["role"] = role
    df["freq_protocol_used"] = protocol
    return df

In [131]:
# ------------------------------------------------------------
# RRHO (rigid-rotor harmonic-oscillator) ZPE + vibrational thermal correction.
#
# NOTE: this was referenced below but never actually defined in the original
# notebook -- that's the source of the NameError. Added here.
#
# This computes the *vibrational* contribution only (ZPE + thermal population
# correction), since translational/rotational terms need atomic masses and
# moments of inertia that aren't carried in the frequency CSV. That matches
# how the barrier calculation downstream uses it: it adds zpe_hartree only
# (see "E_total_hartree = E_DFT_hartree + zpe_hartree" below), so the
# reported barriers are ZPE-corrected electronic barriers, not full 298 K
# free-energy barriers. thermal_vib_hartree is still returned for reference /
# for anyone who wants to extend this to a full thermal correction later.
# ------------------------------------------------------------

def rrho_corrections(frequencies_cm1, temperature_k: float = 298.15,
                      imaginary_cutoff_cm1: float = 0.0) -> dict:
    """
    Parameters
    ----------
    frequencies_cm1 : array-like
        Vibrational frequencies in cm^-1. xTB reports imaginary modes as
        negative numbers; those are excluded from the ZPE/thermal sums.
    temperature_k : float
        Temperature for the vibrational population correction (default 298.15 K).
    imaginary_cutoff_cm1 : float
        Frequencies <= this value are treated as imaginary/excluded. Kept at
        0.0 rather than a small positive buffer so that genuine (if small)
        real low-frequency modes aren't accidentally dropped; xTB's own
        Hessian projection already removes translation/rotation.

    Returns
    -------
    dict with:
        zpe_hartree          : zero-point energy (real modes only)
        thermal_vib_hartree  : vibrational thermal population correction, U_vib(T) - ZPE
        n_imaginary          : count of imaginary (<= cutoff) modes
        n_real               : count of real modes used
        n_total              : total modes seen
    """
    freqs = np.asarray(frequencies_cm1, dtype=float)
    is_imaginary = freqs <= imaginary_cutoff_cm1
    real_freqs = freqs[~is_imaginary]

    # E_i = h c nu_i, per mode (J)
    energies_j = H_PLANCK * C_LIGHT_CM * real_freqs

    # ZPE = sum(1/2 h c nu_i), per mole, then to hartree
    zpe_j_per_mol = np.sum(0.5 * energies_j) * N_AVOGADRO
    zpe_hartree = zpe_j_per_mol / (HARTREE_TO_KJMOL * 1000.0)

    # Vibrational thermal population correction: sum_i [E_i / (exp(E_i/kT) - 1)]
    kT = K_BOLTZ * temperature_k
    x = energies_j / kT if kT > 0 else np.full_like(energies_j, np.inf)
    x_clipped = np.minimum(x, 700.0)  # avoid overflow in exp for high-freq modes
    pop_term = np.where(x > 700.0, 0.0, energies_j / (np.exp(x_clipped) - 1.0))
    thermal_vib_j_per_mol = np.sum(pop_term) * N_AVOGADRO
    thermal_vib_hartree = thermal_vib_j_per_mol / (HARTREE_TO_KJMOL * 1000.0)

    return {
        "zpe_hartree": zpe_hartree,
        "thermal_vib_hartree": thermal_vib_hartree,
        "n_imaginary": int(np.sum(is_imaginary)),
        "n_real": int(np.sum(~is_imaginary)),
        "n_total": int(freqs.size),
    }


In [132]:

freq_df = load_all_xtb_frequencies(STEPS, ROLES)
freq_df.head()

thermo_rows = []
for (step, role), grp in freq_df.groupby(["step", "role"]):
    corr = rrho_corrections(grp["frequency_cm1"].values)
    thermo_rows.append({"step": step, "role": role, **corr})
thermo_df = pd.DataFrame(thermo_rows)

# Sanity check: RS/PS (minima) should have 0 imaginary modes; TS (saddle
# points) should have exactly 1. GFN2_etemp500 smearing (used as a fallback
# for a few steps -- see FREQ_PROTOCOL_OVERRIDES above) can occasionally
# shift a very soft mode across zero, so flag anything unexpected rather
# than silently trusting it.
def _expected_n_imaginary(role):
    return 1 if role == "TS" else 0

thermo_df["n_imaginary_expected"] = thermo_df["role"].map(_expected_n_imaginary)
thermo_df["imaginary_mode_ok"] = thermo_df["n_imaginary"] == thermo_df["n_imaginary_expected"]

bad_modes = thermo_df[~thermo_df["imaginary_mode_ok"]]
if len(bad_modes):
    print(f"WARNING: {len(bad_modes)} step/role(s) have an unexpected imaginary-mode count "
          f"(check these Hessians before trusting the barrier):")
    display(bad_modes[["step", "role", "n_imaginary", "n_imaginary_expected"]])

thermo_df


,step,role,n_imaginary,n_imaginary_expected
1,RS1_1_to_TS1_2,RS,1,0
4,RS1_2b_to_PS1_3,RS,1,0
5,RS1_2b_to_PS1_3,TS,0,1
7,RS2_1_to_TS2_1a,RS,1,0
8,RS2_1_to_TS2_1a,TS,0,1
11,TS1_2_to_PS1_2b,TS,0,1


,step,role,zpe_hartree,thermal_vib_hartree,n_imaginary,n_real,n_total,n_imaginary_expected,imaginary_mode_ok
0,RS1_1_to_TS1_2,PS,0.467528,0.031817,0,165,165,0,True
1,RS1_1_to_TS1_2,RS,0.467286,0.031021,1,164,165,0,False
2,RS1_1_to_TS1_2,TS,0.467502,0.030699,1,164,165,1,True
3,RS1_2b_to_PS1_3,PS,0.465543,0.032249,0,165,165,0,True
4,RS1_2b_to_PS1_3,RS,0.469024,0.029995,1,164,165,0,False
5,RS1_2b_to_PS1_3,TS,0.465114,0.032578,0,165,165,1,False
6,RS2_1_to_TS2_1a,PS,0.276681,0.041386,0,165,165,0,True
7,RS2_1_to_TS2_1a,RS,0.280059,0.039370,1,164,165,0,False
8,RS2_1_to_TS2_1a,TS,0.276699,0.041364,0,165,165,1,False
9,TS1_2_to_PS1_2b,PS,0.467417,0.031871,0,165,165,0,True


### Diagnostic: actual imaginary-frequency values for the flagged structures

Fast by design -- reuses `freq_df` already in memory, no file re-reads or
refitting. For each structure with an unexpected imaginary-mode count, shows
the actual imaginary frequency value(s) and the lowest real frequency.

**How to read it:**
- An imaginary frequency with |value| below roughly 20-50 cm^-1 is usually
  just numerical noise from a slightly under-converged optimization or a very
  floppy low-frequency motion (e.g. a methyl/water rotation) sitting right at
  the zero boundary -- often harmless, but worth a tighter re-optimization if
  you want to be rigorous.
- An imaginary frequency with |value| in the hundreds (or more, e.g. proton
  transfers are often 1000-2000i cm^-1) is a REAL mode -- it means that
  geometry is a genuine saddle point along some coordinate. If that shows up
  on something labeled "RS", the RS/TS labeling or the geometry itself is
  wrong, not just numerically noisy.
- If a "TS" shows 0 imaginary modes and the lowest *real* mode is very small
  and floppy, that TS optimization may have relaxed past the saddle point
  into a minimum, or (for the etemp500-fallback cases) smearing may have
  suppressed the mode.

In [133]:
# Fast diagnostic: pulls actual frequency values for flagged structures only,
# straight from freq_df already in memory -- no file I/O, no refitting.
flagged = thermo_df.loc[~thermo_df["imaginary_mode_ok"], ["step", "role"]]

diag_rows = []
for _, fr in flagged.iterrows():
    step, role = fr["step"], fr["role"]
    sub = freq_df[(freq_df["step"] == step) & (freq_df["role"] == role)]
    freqs_sorted = np.sort(sub["frequency_cm1"].values)
    imag = freqs_sorted[freqs_sorted <= 0.0]
    real = freqs_sorted[freqs_sorted > 0.0]
    diag_rows.append({
        "step": step,
        "role": role,
        "protocol_used": sub["freq_protocol_used"].iloc[0] if "freq_protocol_used" in sub.columns else FREQ_PROTOCOL,
        "imaginary_freqs_cm1": [round(f, 1) for f in imag],
        "lowest_real_freq_cm1": round(float(real[0]), 1) if len(real) else None,
    })

diag_df = pd.DataFrame(diag_rows)
diag_df


,step,role,protocol_used,imaginary_freqs_cm1,lowest_real_freq_cm1
0,RS1_1_to_TS1_2,RS,GFN2_tight,[-3.0],10.1
1,RS1_2b_to_PS1_3,RS,GFN2_etemp500,[-9.6],11.3
2,RS1_2b_to_PS1_3,TS,GFN2_etemp500,[],10.0
3,RS2_1_to_TS2_1a,RS,GFN2_etemp500,[-5.7],8.5
4,RS2_1_to_TS2_1a,TS,GFN2_etemp500,[],5.7
5,TS1_2_to_PS1_2b,TS,GFN2_tight,[],12.6


## 7b. FAST PATH -- no ORCA needed at all

Your saved xTB output turned out to be `g98.out` -- xtb's Gaussian-format
file for visualization tools, not the raw console transcript, so it doesn't
reliably contain a parseable `TOTAL ENERGY` line. Rather than guess at
another filename, this runs a **fresh xtb single point** on each
already-optimized geometry.

**This is still fast and does NOT redo the expensive part** -- the geometry
optimization and Hessian are already done; a single-point energy call on an
already-converged 40-80 atom structure typically takes a few seconds. 15
structures should take well under a minute total.

**Trade-off, unchanged from before:** GFN2-xTB electronic energy (matched to
whichever protocol/electronic-temperature setting produced that structure's
ZPE, for internal consistency) instead of DFT, and Mulliken charges instead
of RESP.

Off by default -- flip `RUN_FAST_XTB_SP = True` once `XTB_EXE` below points
to your actual xtb executable (if `xtb` is already on your PATH, the default
`"xtb"` will work as-is).

In [134]:
import subprocess, re, json

RUN_FAST_XTB_SP = False   # <-- flip to True to actually run the fast single points
XTB_EXE = "xtb"           # <-- full path if xtb isn't on PATH, e.g. r"C:\\xtb\\bin\\xtb.exe"

# Matches the same protocol (and electronic temperature, if any) that produced
# each structure's Hessian/ZPE, so the electronic energy and thermal
# correction are computed at consistent settings.
_PROTOCOL_TO_XTB_FLAGS = {
    "GFN1_tight":     ["--gfn", "1"],
    "GFN2_tight":     ["--gfn", "2"],
    "GFN2_normal":    ["--gfn", "2"],
    "GFN2_etemp300":  ["--gfn", "2", "--etemp", "300"],
    "GFN2_etemp500":  ["--gfn", "2", "--etemp", "500"],
}

FAST_XTB_SP_DIR = STAGE2_XTB_DIR.parent / "fast_xtb_sp_reruns"

def parse_xtb_total_energy_from_text(text: str) -> float | None:
    for line in text.splitlines():
        if "TOTAL ENERGY" in line.upper():
            nums = re.findall(r"-?\d+\.\d+", line)
            if nums:
                return float(nums[0])
    return None

def parse_xtb_mulliken_charges_from_text(text: str, n_atoms: int) -> dict | None:
    lines = text.splitlines()
    start = None
    for i, line in enumerate(lines):
        if "Mulliken" in line and "charge" in line.lower():
            start = i + 1
            break
    if start is None:
        return None
    charges = {}
    for j in range(start, min(start + n_atoms + 5, len(lines))):
        parts = lines[j].split()
        if len(parts) < 3:
            if charges:
                break
            continue
        try:
            idx = int(parts[0]); elem = parts[1]; val = float(parts[2])
            charges[f"{idx}_{elem}"] = val
        except (ValueError, IndexError):
            if charges:
                break
    return charges or None

def run_xtb_singlepoint(xyz_path: Path, work_dir: Path, charge: int, multiplicity: int,
                         protocol: str, xtb_exe: str = XTB_EXE) -> dict:
    work_dir.mkdir(parents=True, exist_ok=True)
    flags = _PROTOCOL_TO_XTB_FLAGS.get(protocol, ["--gfn", "2"])
    cmd = [xtb_exe, str(xyz_path), "--chrg", str(charge), "--uhf", str(multiplicity - 1)] + flags
    result = subprocess.run(cmd, cwd=work_dir, capture_output=True, text=True)
    (work_dir / "xtb_sp_stdout.log").write_text(result.stdout, encoding="utf-8")
    if result.returncode != 0:
        return {"E_xtb_hartree": None, "mulliken_charges_json": None,
                "note": f"xtb returncode={result.returncode}: {result.stderr[:300]}"}
    energy = parse_xtb_total_energy_from_text(result.stdout)
    labels, _, _ = read_xyz(xyz_path)
    charges = parse_xtb_mulliken_charges_from_text(result.stdout, len(labels))
    return {
        "E_xtb_hartree": energy,
        "mulliken_charges_json": json.dumps(charges) if charges else None,
        "note": "" if energy is not None else "xtb ran but 'TOTAL ENERGY' line not matched -- check xtb_sp_stdout.log",
    }

fast_rows = []
for step in STEPS:
    cfg = STRUCTURE_SETS[step]
    for role in ROLES:
        xyz_path = XTB_DIR / step / f"{step}_{role}.xyz"
        protocol = FREQ_PROTOCOL_OVERRIDES.get((step, role), FREQ_PROTOCOL)
        work_dir = FAST_XTB_SP_DIR / step / role
        row = {"step": step, "role": role, "protocol_used": protocol}
        if not xyz_path.exists():
            row.update({"E_xtb_hartree": None, "mulliken_charges_json": None, "note": f"missing xyz: {xyz_path}"})
            fast_rows.append(row)
            continue
        if not RUN_FAST_XTB_SP:
            row.update({"E_xtb_hartree": None, "mulliken_charges_json": None,
                        "note": "RUN_FAST_XTB_SP is False -- not running"})
            fast_rows.append(row)
            continue
        print(f"  fast xtb SP: {step}/{role} ({protocol}) ...", flush=True)
        result = run_xtb_singlepoint(xyz_path, work_dir, cfg.charge, cfg.multiplicity, protocol)
        row.update(result)
        fast_rows.append(row)

fast_energies_df = pd.DataFrame(fast_rows)
fast_energies_df.to_csv(SUMMARY_OUT / "xtb_fast_energies_charges.csv", index=False)
n_found = fast_energies_df["E_xtb_hartree"].notna().sum()
print(f"\nGot energy for {n_found}/{len(fast_energies_df)} structures.")
fast_energies_df



Got energy for 0/15 structures.


,step,role,protocol_used,E_xtb_hartree,mulliken_charges_json,note
0,RS1_1_to_TS1_2,RS,GFN2_tight,None,None,RUN_FAST_XTB_SP is False -- not running
1,RS1_1_to_TS1_2,TS,GFN2_tight,None,None,RUN_FAST_XTB_SP is False -- not running
2,RS1_1_to_TS1_2,PS,GFN2_tight,None,None,RUN_FAST_XTB_SP is False -- not running
3,TS1_2_to_PS1_2b,RS,GFN2_tight,None,None,RUN_FAST_XTB_SP is False -- not running
4,TS1_2_to_PS1_2b,TS,GFN2_tight,None,None,RUN_FAST_XTB_SP is False -- not running
5,TS1_2_to_PS1_2b,PS,GFN2_etemp500,None,None,RUN_FAST_XTB_SP is False -- not running
6,RS1_2b_to_PS1_3,RS,GFN2_etemp500,None,None,RUN_FAST_XTB_SP is False -- not running
7,RS1_2b_to_PS1_3,TS,GFN2_etemp500,None,None,RUN_FAST_XTB_SP is False -- not running
8,RS1_2b_to_PS1_3,PS,GFN2_etemp500,None,None,RUN_FAST_XTB_SP is False -- not running
9,RS2_1_to_TS2_1a,RS,GFN2_etemp500,None,None,RUN_FAST_XTB_SP is False -- not running


In [135]:
# Self-contained (does NOT depend on compute_barriers() defined later in
# section 8) -- same TS-RS / PS-RS logic, just using xTB electronic energies.
combined_fast = fast_energies_df.merge(thermo_df, on=["step", "role"], how="left")
combined_fast["E_total_hartree"] = combined_fast["E_xtb_hartree"] + combined_fast["zpe_hartree"]

def _compute_barriers_fast(combined: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for step, grp in combined.groupby("step"):
        g = grp.set_index("role")
        if not {"RS", "TS", "PS"}.issubset(g.index):
            continue
        if g["E_total_hartree"].isna().any():
            rows.append({"step": step, "barrier_kcal_mol": None, "reaction_energy_kcal_mol": None,
                         "note": "missing xtb energy -- check log parsing above"})
            continue
        barrier = (g.loc["TS", "E_total_hartree"] - g.loc["RS", "E_total_hartree"]) * HARTREE_TO_KCAL
        rxn_energy = (g.loc["PS", "E_total_hartree"] - g.loc["RS", "E_total_hartree"]) * HARTREE_TO_KCAL
        rows.append({"step": step, "barrier_kcal_mol": barrier, "reaction_energy_kcal_mol": rxn_energy,
                     "note": "ZPE-corrected, GFN2-xTB-el//xTB-thermal (FAST PATH, no DFT)"})
    return pd.DataFrame(rows)

barriers_fast_df = _compute_barriers_fast(combined_fast)
barriers_fast_df.to_csv(SUMMARY_OUT / "barriers_fast_xtb.csv", index=False)
barriers_fast_df


,step,barrier_kcal_mol,reaction_energy_kcal_mol,note
0,RS1_1_to_TS1_2,None,None,missing xtb energy -- check log parsing above
1,RS1_2b_to_PS1_3,None,None,missing xtb energy -- check log parsing above
2,RS2_1_to_TS2_1a,None,None,missing xtb energy -- check log parsing above
3,TS1_2_to_PS1_2b,None,None,missing xtb energy -- check log parsing above
4,TS2_1a_to_PS2_2,None,None,missing xtb energy -- check log parsing above


## 8. Combine DFT electronic energy + xTB thermal corrections -> barriers

In [136]:

combined = dft_energies.merge(thermo_df, on=["step", "role"], how="left")
combined["E_total_hartree"] = combined["E_DFT_hartree"] + combined["zpe_hartree"]

def compute_barriers(combined: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for step, grp in combined.groupby("step"):
        g = grp.set_index("role")
        if not {"RS", "TS", "PS"}.issubset(g.index):
            continue
        if g["E_total_hartree"].isna().any():
            rows.append({"step": step, "barrier_kcal_mol": None, "reaction_energy_kcal_mol": None,
                         "note": "missing energy -- check DFT job completion"})
            continue
        barrier = (g.loc["TS", "E_total_hartree"] - g.loc["RS", "E_total_hartree"]) * HARTREE_TO_KCAL
        rxn_energy = (g.loc["PS", "E_total_hartree"] - g.loc["RS", "E_total_hartree"]) * HARTREE_TO_KCAL
        rows.append({"step": step, "barrier_kcal_mol": barrier,
                     "reaction_energy_kcal_mol": rxn_energy, "note": "ZPE-corrected, DFT-el//xTB-thermal"})
    return pd.DataFrame(rows)

barriers_df = compute_barriers(combined)
barriers_df.to_csv(SUMMARY_OUT / "barriers_zpe_corrected.csv", index=False)
barriers_df


,step,barrier_kcal_mol,reaction_energy_kcal_mol,note
0,RS1_1_to_TS1_2,None,None,missing energy -- check DFT job completion
1,RS1_2b_to_PS1_3,None,None,missing energy -- check DFT job completion
2,RS2_1_to_TS2_1a,None,None,missing energy -- check DFT job completion
3,TS1_2_to_PS1_2b,None,None,missing energy -- check DFT job completion
4,TS2_1a_to_PS2_2,None,None,missing energy -- check DFT job completion


## 9. RESP charge fitting (two-stage, Bayly et al. 1993) — no AmberTools

In [137]:

def _fibonacci_sphere(n_points: int, radius: float, center: np.ndarray) -> np.ndarray:
    if n_points < 1:
        return np.zeros((0, 3))
    indices = np.arange(0, n_points, dtype=float) + 0.5
    phi = np.arccos(1 - 2 * indices / n_points)
    golden_angle = np.pi * (3.0 - np.sqrt(5.0))
    theta = golden_angle * indices
    x, y, z = np.sin(phi)*np.cos(theta), np.sin(phi)*np.sin(theta), np.cos(phi)
    return np.stack([x, y, z], axis=1) * radius + center


def generate_mk_esp_grid(labels, coords_angstrom, layers=(1.4, 1.6, 1.8, 2.0), density_per_ang2=1.0):
    coords = np.asarray(coords_angstrom, dtype=float)
    radii = np.array([VDW_RADII_ANGSTROM.get(lab.upper(), 1.70) for lab in labels])
    kept = []
    for scale in layers:
        scaled_radii = radii * scale
        for i in range(len(labels)):
            r_i = scaled_radii[i]
            n_pts = max(1, int(round(4 * np.pi * r_i**2 * density_per_ang2)))
            candidates = _fibonacci_sphere(n_pts, r_i, coords[i])
            other_idx = [j for j in range(len(labels)) if j != i]
            if other_idx:
                other_coords = coords[other_idx]
                other_radii = scaled_radii[other_idx]
                dists = np.linalg.norm(candidates[:, None, :] - other_coords[None, :, :], axis=2)
                outside_all = np.all(dists >= other_radii[None, :], axis=1)
                candidates = candidates[outside_all]
            kept.append(candidates)
    return np.concatenate(kept, axis=0) if kept else np.zeros((0, 3))


def write_grid_for_orca_vpot(grid_angstrom: np.ndarray, path: Path) -> None:
    '''Verify exact format/units against `orca_vpot` (no args) for your ORCA version.'''
    grid_bohr = grid_angstrom * ANGSTROM_TO_BOHR
    lines = [str(len(grid_bohr))] + [f"{x:16.10f} {y:16.10f} {z:16.10f}" for x, y, z in grid_bohr]
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text("\n".join(lines) + "\n", encoding="utf-8")


def read_vpot_grid(path: Path) -> tuple[np.ndarray, np.ndarray]:
    """Returns (points_bohr, potential_hartree). Returns empty (0,3)/(0,)
    arrays -- never raises -- if the file has no parseable rows, so callers
    can check `.size == 0` and skip cleanly instead of crashing on indexing
    an empty array. Handles Fortran 'D' exponent notation (e.g. 1.234D+02)
    in case orca_vpot or a hand-edited file uses it."""
    rows = []
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        parts = line.split()
        if len(parts) < 4:
            continue
        try:
            vals = [float(p.replace("D", "E").replace("d", "e")) for p in parts[:4]]
            rows.append(vals)
        except ValueError:
            continue
    if not rows:
        return np.zeros((0, 3)), np.zeros((0,))
    arr = np.array(rows)
    return arr[:, :3], arr[:, 3]


In [138]:

@dataclass
class RespResult:
    charges: np.ndarray
    stage1_charges: np.ndarray
    rmse_hartree: float
    equivalence_groups: list = field(default_factory=list)


def _potential_matrix(points_bohr, atom_coords_bohr):
    diff = points_bohr[:, None, :] - atom_coords_bohr[None, :, :]
    dist = np.maximum(np.linalg.norm(diff, axis=2), 1e-3)
    return 1.0 / dist


def _restrained_objective(q_free, group_of_atom, n_groups, A, V_qm, restrained_mask, a_r, b_r):
    q_atom = q_free[group_of_atom]
    esp_term = np.sum((A @ q_atom - V_qm) ** 2)
    restraint = sum(a_r * (np.sqrt(q_free[g]**2 + b_r**2) - b_r) for g in range(n_groups) if restrained_mask[g])
    return esp_term + restraint


def _fit_stage(A, V_qm, group_of_atom, n_groups, net_charge, restrained_mask, a_r, b_r,
               q0=None, fixed_groups=None):
    fixed_groups = fixed_groups or {}
    free_groups = [g for g in range(n_groups) if g not in fixed_groups]
    q0_free = np.zeros(len(free_groups)) if q0 is None else q0[free_groups]
    fixed_sum = sum(fixed_groups.values())
    q_template = np.zeros(n_groups)
    for g, val in fixed_groups.items():
        q_template[g] = val

    def objective(qv):
        q_full = q_template.copy(); q_full[free_groups] = qv
        return _restrained_objective(q_full, group_of_atom, n_groups, A, V_qm, restrained_mask, a_r, b_r)

    def constraint(qv):
        return np.sum(qv) - (net_charge - fixed_sum)

    res = minimize(objective, q0_free, method="SLSQP",
                    constraints=[{"type": "eq", "fun": constraint}],
                    options={"maxiter": 2000, "ftol": 1e-12})
    if not res.success:
        raise RuntimeError(f"RESP fit did not converge: {res.message}")
    q_full = q_template.copy(); q_full[free_groups] = res.x
    return q_full


def resp_fit_two_stage(labels, atom_coords_angstrom, grid_points_bohr, grid_potential_hartree,
                        net_charge, equivalence_groups=None, stage2_ch_atom_indices=None):
    n_atoms = len(labels)
    coords_bohr = np.asarray(atom_coords_angstrom, dtype=float) * ANGSTROM_TO_BOHR
    A = _potential_matrix(grid_points_bohr, coords_bohr)

    equivalence_groups = equivalence_groups or []
    grouped = set(a for grp in equivalence_groups for a in grp)
    groups = list(equivalence_groups) + [[a] for a in range(n_atoms) if a not in grouped]
    n_groups = len(groups)
    group_of_atom = np.zeros(n_atoms, dtype=int)
    for g, atoms in enumerate(groups):
        for a in atoms:
            group_of_atom[a] = g

    q_stage1 = _fit_stage(A, grid_potential_hartree, group_of_atom, n_groups, net_charge,
                           np.ones(n_groups, dtype=bool), RESP_A_STAGE1, RESP_B)
    atom_charges_stage1 = q_stage1[group_of_atom]

    stage2_ch_atom_indices = stage2_ch_atom_indices or []
    stage2_groups = sorted(set(group_of_atom[a] for a in stage2_ch_atom_indices))
    if stage2_groups:
        fixed = {g: q_stage1[g] for g in range(n_groups) if g not in stage2_groups}
        mask2 = np.zeros(n_groups, dtype=bool)
        for g in stage2_groups:
            mask2[g] = True
        q_final = _fit_stage(A, grid_potential_hartree, group_of_atom, n_groups, net_charge,
                              mask2, RESP_A_STAGE2, RESP_B, q0=q_stage1, fixed_groups=fixed)
    else:
        q_final = q_stage1

    atom_charges_final = q_final[group_of_atom]
    rmse = float(np.sqrt(np.mean((A @ atom_charges_final - grid_potential_hartree) ** 2)))
    return RespResult(atom_charges_final, atom_charges_stage1, rmse, groups)


def guess_stage2_ch_indices(labels, coords_angstrom, bond_cutoff=1.8,
                             polar_elements=("N", "O", "S", "P", "F", "CL", "BR")):
    coords = np.asarray(coords_angstrom, dtype=float)
    elements = [l.upper() for l in labels]
    n = len(labels)
    dist = np.linalg.norm(coords[:, None, :] - coords[None, :, :], axis=2)
    bonded = (dist < bond_cutoff) & (dist > 1e-6)
    def neighbors(i): return [j for j in range(n) if bonded[i, j]]
    flagged = []
    for i, el in enumerate(elements):
        if el != "H":
            continue
        nbrs = neighbors(i)
        if len(nbrs) != 1 or elements[nbrs[0]] != "C":
            continue
        carbon_nbrs = neighbors(nbrs[0])
        if any(elements[j] in polar_elements for j in carbon_nbrs):
            continue
        flagged.append(i)
    return flagged


### Self-test (synthetic two-point-charge system) — run once to verify the fit math

In [139]:

def _self_test():
    rng = np.random.default_rng(0)
    true_coords_bohr = np.array([[0.0, 0.0, 0.0], [2.0, 0.0, 0.0]])
    true_charges = np.array([0.5, -0.5])
    pts = rng.uniform(-4, 6, size=(400, 3))
    A = _potential_matrix(pts, true_coords_bohr)
    V = A @ true_charges
    coords_ang = true_coords_bohr / ANGSTROM_TO_BOHR
    result = resp_fit_two_stage(["C", "C"], coords_ang, pts, V, net_charge=0)
    assert np.allclose(result.charges, true_charges, atol=1e-3), "RESP self-test FAILED"
    print("RESP self-test PASSED:", result.charges)

_self_test()


RESP self-test PASSED: [ 0.49988791 -0.49988791]


## 10. Run RESP fitting on your real structures

After running the `_resp_hf631gs.inp` jobs and `orca_vpot` on each, point
this loop at the resulting `.gbw`/vpot output files. This example assumes
one `vpot_out.txt` per step/role sitting next to its RESP input file —
adjust the path pattern to match whatever `orca_vpot` actually produced.

In [140]:
import traceback

resp_rows = []
resp_csv_path = SUMMARY_OUT / "resp_charges.csv"

def _save_resp_checkpoint():
    """Write whatever has completed so far. Called after every structure,
    so a crash partway through never costs you the results already fitted."""
    pd.DataFrame(resp_rows).to_csv(resp_csv_path, index=False)

for _, r in sp_manifest.iterrows():
    step, role = r["step"], r["role"]
    print(f"RESP fit: {step} / {role} ...", flush=True)
    row = {"step": step, "role": role, "resp_done": False}
    try:
        xyz_path = Path(r["xyz_source"])
        labels, coords, _ = read_xyz(xyz_path)

        grid = generate_mk_esp_grid(labels, coords)
        grid_file = Path(r["orca_resp"]).with_name(f"{step}_{role}_grid.txt")
        write_grid_for_orca_vpot(grid, grid_file)

        vpot_file = Path(r["orca_resp"]).with_name(f"{step}_{role}_vpot_out.txt")  # <-- adjust to actual orca_vpot output name
        if not vpot_file.exists():
            row["note"] = "vpot output file not found"
            resp_rows.append(row)
            _save_resp_checkpoint()
            continue

        points_bohr, V = read_vpot_grid(vpot_file)

        # Validate BEFORE handing anything to scipy's SLSQP (compiled Fortran
        # code): malformed/empty/mismatched/non-finite input here is a very
        # plausible cause of a native crash that Python can't catch cleanly,
        # unlike a normal exception. Catching it here turns a kernel-killing
        # crash into a logged, skippable row.
        if points_bohr.size == 0 or V.size == 0:
            row["note"] = f"vpot file parsed to 0 valid rows -- check format of {vpot_file.name}"
            resp_rows.append(row)
            _save_resp_checkpoint()
            continue
        if points_bohr.shape[0] != V.shape[0]:
            row["note"] = f"grid/potential row count mismatch ({points_bohr.shape[0]} vs {V.shape[0]})"
            resp_rows.append(row)
            _save_resp_checkpoint()
            continue
        if not (np.all(np.isfinite(points_bohr)) and np.all(np.isfinite(V))):
            row["note"] = "NaN/Inf in parsed vpot grid or potential -- check vpot file for truncation/parse errors"
            resp_rows.append(row)
            _save_resp_checkpoint()
            continue

        ch_indices = guess_stage2_ch_indices(labels, coords)
        result = resp_fit_two_stage(labels, coords, points_bohr, V,
                                     net_charge=r["charge"], stage2_ch_atom_indices=ch_indices)

        charge_json = json.dumps({f"{i}_{lab}": float(q) for i, (lab, q) in enumerate(zip(labels, result.charges))})
        row.update({
            "resp_done": True,
            "resp_rmse_hartree": result.rmse_hartree,
            "resp_charges_json": charge_json,
        })
    except Exception as e:
        row["note"] = f"{type(e).__name__}: {e}"
        traceback.print_exc()

    resp_rows.append(row)
    _save_resp_checkpoint()  # <-- written after EVERY structure, success or failure

resp_df = pd.DataFrame(resp_rows)
resp_df.to_csv(resp_csv_path, index=False)
resp_df


RESP fit: RS1_1_to_TS1_2 / RS ...
RESP fit: RS1_1_to_TS1_2 / TS ...
RESP fit: RS1_1_to_TS1_2 / PS ...
RESP fit: TS1_2_to_PS1_2b / RS ...
RESP fit: TS1_2_to_PS1_2b / TS ...
RESP fit: TS1_2_to_PS1_2b / PS ...
RESP fit: RS1_2b_to_PS1_3 / RS ...
RESP fit: RS1_2b_to_PS1_3 / TS ...
RESP fit: RS1_2b_to_PS1_3 / PS ...
RESP fit: RS2_1_to_TS2_1a / RS ...
RESP fit: RS2_1_to_TS2_1a / TS ...
RESP fit: RS2_1_to_TS2_1a / PS ...
RESP fit: TS2_1a_to_PS2_2 / RS ...
RESP fit: TS2_1a_to_PS2_2 / TS ...
RESP fit: TS2_1a_to_PS2_2 / PS ...


,step,role,resp_done,note
0,RS1_1_to_TS1_2,RS,False,vpot output file not found
1,RS1_1_to_TS1_2,TS,False,vpot output file not found
2,RS1_1_to_TS1_2,PS,False,vpot output file not found
3,TS1_2_to_PS1_2b,RS,False,vpot output file not found
4,TS1_2_to_PS1_2b,TS,False,vpot output file not found
5,TS1_2_to_PS1_2b,PS,False,vpot output file not found
6,RS1_2b_to_PS1_3,RS,False,vpot output file not found
7,RS1_2b_to_PS1_3,TS,False,vpot output file not found
8,RS1_2b_to_PS1_3,PS,False,vpot output file not found
9,RS2_1_to_TS2_1a,RS,False,vpot output file not found


## 11. Final combined EVB-reference summary

In [141]:

final_summary = combined.merge(resp_df, on=["step", "role"], how="left")
final_summary = final_summary.merge(barriers_df, on="step", how="left")
final_summary.to_csv(SUMMARY_OUT / "evb_reference_summary_FULL.csv", index=False)
final_summary


,step,role,orca_out,found,E_DFT_hartree,zpe_hartree,thermal_vib_hartree,n_imaginary,n_real,n_total,n_imaginary_expected,imaginary_mode_ok,E_total_hartree,resp_done,note_x,barrier_kcal_mol,reaction_energy_kcal_mol,note_y
0,RS1_1_to_TS1_2,RS,D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_5QM\s...,False,None,0.467286,0.031021,1,164,165,0,False,NaN,False,vpot output file not found,None,None,missing energy -- check DFT job completion
1,RS1_1_to_TS1_2,TS,D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_5QM\s...,False,None,0.467502,0.030699,1,164,165,1,True,NaN,False,vpot output file not found,None,None,missing energy -- check DFT job completion
2,RS1_1_to_TS1_2,PS,D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_5QM\s...,False,None,0.467528,0.031817,0,165,165,0,True,NaN,False,vpot output file not found,None,None,missing energy -- check DFT job completion
3,TS1_2_to_PS1_2b,RS,D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_5QM\s...,False,None,0.467856,0.031366,0,165,165,0,True,NaN,False,vpot output file not found,None,None,missing energy -- check DFT job completion
4,TS1_2_to_PS1_2b,TS,D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_5QM\s...,False,None,0.467859,0.031368,0,165,165,1,False,NaN,False,vpot output file not found,None,None,missing energy -- check DFT job completion
5,TS1_2_to_PS1_2b,PS,D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_5QM\s...,False,None,0.467417,0.031871,0,165,165,0,True,NaN,False,vpot output file not found,None,None,missing energy -- check DFT job completion
6,RS1_2b_to_PS1_3,RS,D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_5QM\s...,False,None,0.469024,0.029995,1,164,165,0,False,NaN,False,vpot output file not found,None,None,missing energy -- check DFT job completion
7,RS1_2b_to_PS1_3,TS,D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_5QM\s...,False,None,0.465114,0.032578,0,165,165,1,False,NaN,False,vpot output file not found,None,None,missing energy -- check DFT job completion
8,RS1_2b_to_PS1_3,PS,D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_5QM\s...,False,None,0.465543,0.032249,0,165,165,0,True,NaN,False,vpot output file not found,None,None,missing energy -- check DFT job completion
9,RS2_1_to_TS2_1a,RS,D:\PhD_Thesis\LmrR_EVB\charges\fast_lmrr_5QM\s...,False,None,0.280059,0.039370,1,164,165,0,False,NaN,False,vpot output file not found,None,None,missing energy -- check DFT job completion


## 12. Where this leaves you for EVB

At this point `evb_reference_summary_FULL.csv` contains, per step:
- **QM electronic energy** (PBE0-D3BJ single point on the xTB geometry)
- **ZPE + thermal correction** (from your existing xTB Hessians)
- **ZPE-corrected activation barrier and reaction energy** (kcal/mol)
- **RESP atomic charges** for RS/TS/PS (HF/6-31G\*, AMBER convention)
- **Normal-mode count / imaginary-mode check**, inherited from your Stage-1/2
  frequency analysis (already confirms TS validity — keep that check as-is)

This is a legitimate, properly-leveled QM reference dataset. It is **not**
yet an EVB free energy surface. The next stage — outside the scope of a
QM-only notebook — is building the classical valence-bond Hamiltonian
(diabatic RS/PS states parameterized with these RESP charges), fitting the
EVB off-diagonal coupling and shift parameters to reproduce this QM barrier
and reaction energy, and running FEP/umbrella-sampling MD in the solvated
protein to get the actual condensed-phase free-energy barrier. That step
needs an MD/EVB engine (e.g. Q, CHARMM, or an OpenMM-based EVB setup), and
is worth scoping separately once this QM reference set is complete and
you've sanity-checked the barriers against your original xTB pre-screening
values (they should shift, but not wildly, if xTB was a reasonable proxy).
